
# BigQuery Advanced Lab

ShopIndex (a small e-commerce company) has outgrown basic load-query-join usage of BigQuery. Some
data lives outside it entirely, the bill is unpredictable, an audit needs proof of who can see
what, and the data team wants models and a semantic search feature without ever exporting data
out of the warehouse. This notebook works through all of that against one running dataset —
`customers`, `products`, `orders` — instead of switching context every section.

Covers: external tables & Lakehouse for Iceberg, pricing/reservations, IAM, BQML (4 model types),
time travel, Vector Search for RAG, remote functions, data lineage, RLS, and encryption (AEAD +
masking).

**Terminology note:** "BigLake" for Iceberg tables was renamed "Lakehouse for Apache Iceberg" in
April 2026 — the `WITH CONNECTION` syntax and IAM roles didn't change, just the console name.

**Cost note:** most of this notebook is cheap (a few cents in on-demand query cost). BQML
training, Vertex AI embeddings, and slot reservations are billed beyond that, which is why each of
those sections is gated behind its own `RUN_*_STEP` toggle below — **all default to False**, flip
one on right before you get to that section, not all at once at the top.


## Config

In [ ]:
PROJECT_ID = "your-gcp-project-id"  #@param {type:"string"}
BUCKET_NAME = "your-gcs-bucket-name"  #@param {type:"string"}
DATASET_ID = "ecommerce"  #@param {type:"string"}
LOCATION = "US"  #@param {type:"string"}

# Leave these off until you're actually at that section -- each one is a real, billed operation.
RUN_BQML_STEP = False              #@param {type:"boolean"}
RUN_BQML_FORECAST_STEP = False     #@param {type:"boolean"}
RUN_VECTOR_SEARCH_STEP = False     #@param {type:"boolean"}
RUN_RLS_STEP = False               #@param {type:"boolean"}
RUN_ICEBERG_STEP = False           #@param {type:"boolean"}
RUN_RESERVATION_STEP = False       #@param {type:"boolean"}

def TBL(name):
    return f"`{PROJECT_ID}.{DATASET_ID}.{name}`"

print(f"Project: {PROJECT_ID} | Bucket: {BUCKET_NAME} | Dataset: {DATASET_ID} | Location: {LOCATION}")


## Auth & clients

In [ ]:
from google.colab import auth
auth.authenticate_user()
print("authenticated")


In [ ]:
!pip install --quiet google-cloud-bigquery google-cloud-storage google-cloud-bigquery-reservation google-cloud-datacatalog-lineage

from google.cloud import bigquery
from google.cloud import storage
from google.cloud.exceptions import Conflict
import pandas as pd

bq_client = bigquery.Client(project=PROJECT_ID)
storage_client = storage.Client(project=PROJECT_ID)

def run_query(sql, label=None, expect_rows=True):
    if label:
        print(f"--- {label} ---")
    job = bq_client.query(sql)
    if expect_rows:
        try:
            display(job.result().to_dataframe())
        except Exception:
            job.result()
    else:
        job.result()
    if job.total_bytes_processed is not None:
        print(f"bytes processed: {job.total_bytes_processed:,}")
    return job

print("clients ready")


In [ ]:
!gcloud services enable bigquery.googleapis.com bigqueryconnection.googleapis.com \
  bigqueryreservation.googleapis.com datalineage.googleapis.com dataplex.googleapis.com \
  --project={PROJECT_ID}

dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = LOCATION
try:
    bq_client.create_dataset(dataset_ref)
    print(f"created dataset {DATASET_ID}")
except Conflict:
    print(f"{DATASET_ID} already exists — reusing it")

bucket = storage_client.bucket(BUCKET_NAME)
if not bucket.exists():
    bucket = storage_client.create_bucket(BUCKET_NAME, location=LOCATION)
    print(f"created bucket {BUCKET_NAME}")
else:
    print(f"{BUCKET_NAME} already exists — reusing it")



> 🖥️ SQL workspace's Explorer shows the new dataset; Cloud Storage > Buckets shows the bucket.



## 1. Native tables
`orders` has a nested `line_items` array (the basis for every UNNEST later) and is already
partitioned + clustered — that's Section 7's subject, set up now so it's there when we need it.


In [ ]:
run_query(f'''
CREATE OR REPLACE TABLE {TBL("customers")} (
  customer_id INT64, first_name STRING, last_name STRING, email STRING,
  city STRING, country STRING, region STRING, customer_segment STRING,
  signup_date DATE, churned BOOL
)
''', label="create customers", expect_rows=False)

run_query(f'''
CREATE OR REPLACE TABLE {TBL("products")} (
  product_id INT64, product_name STRING, category STRING, subcategory STRING,
  unit_price FLOAT64, description STRING
)
''', label="create products", expect_rows=False)

run_query(f'''
CREATE OR REPLACE TABLE {TBL("orders")} (
  order_id INT64, customer_id INT64, region STRING, order_date DATE, status STRING,
  line_items ARRAY<STRUCT<product_id INT64, product_name STRING, quantity INT64, unit_price FLOAT64>>
)
PARTITION BY order_date
CLUSTER BY region, status
''', label="create orders (partitioned + clustered)", expect_rows=False)



> 🖥️ `orders`'s Schema tab shows `line_items` as a `RECORD` with `REPEATED` mode — that's how the
> console renders `ARRAY<STRUCT<...>>`.


## 2. Load the data

In [ ]:
from google.colab import files
print("Upload customers.csv, products.csv, and orders.json (select all three)")
uploaded = files.upload()


In [ ]:
def upload_to_gcs(local_filename):
    blob = bucket.blob(f"{DATASET_ID}/{local_filename}")
    blob.upload_from_filename(local_filename)
    return f"gs://{BUCKET_NAME}/{DATASET_ID}/{local_filename}"

customers_uri = upload_to_gcs("customers.csv")
products_uri = upload_to_gcs("products.csv")
orders_uri = upload_to_gcs("orders.json")
print("uploaded all three to", f"gs://{BUCKET_NAME}/{DATASET_ID}/")


In [ ]:
csv_cfg = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.CSV, skip_leading_rows=1,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
# TBL() returns a backtick-quoted string for embedding in SQL -- these Python client methods
# want a plain "project.dataset.table" string instead, hence .strip("`") everywhere below.
bq_client.load_table_from_uri(customers_uri, TBL("customers").strip("`"), job_config=csv_cfg).result()
bq_client.load_table_from_uri(products_uri, TBL("products").strip("`"), job_config=csv_cfg).result()

json_cfg = bigquery.LoadJobConfig(
    source_format=bigquery.SourceFormat.NEWLINE_DELIMITED_JSON,
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
)
bq_client.load_table_from_uri(orders_uri, TBL("orders").strip("`"), job_config=json_cfg).result()

for name in ["customers", "products", "orders"]:
    print(name, "loaded:", bq_client.get_table(TBL(name).strip("`")).num_rows, "rows")



> 🖥️ `orders`'s Preview tab — expand any row's `line_items` to see the nested array rendered as a
> mini-table, no SQL needed.


## 3. Simple queries

In [ ]:
run_query(f'SELECT customer_id, first_name, last_name, region, customer_segment FROM {TBL("customers")} LIMIT 10',
          label="3a. basic select")

run_query(f'''
SELECT product_name, category, unit_price FROM {TBL("products")}
WHERE category = 'Electronics' AND unit_price > 2000
ORDER BY unit_price DESC
''', label="3b. filter + sort")

run_query(f'''
SELECT region, COUNT(*) AS num_customers, COUNTIF(churned) AS churned_count
FROM {TBL("customers")} GROUP BY region ORDER BY num_customers DESC
''', label="3c. aggregation")



> 🖥️ Paste any of these into the SQL workspace — Job information shows bytes processed, a preview
> of the pricing discussion in Section 8.



## 4. Joins


In [ ]:
run_query(f'''
SELECT o.order_id, o.order_date, o.status, c.first_name, c.last_name, c.region
FROM {TBL("orders")} o JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC LIMIT 15
''', label="4a. inner join")

run_query(f'''
SELECT o.order_id, o.order_date, c.first_name, item.product_name, item.quantity, item.unit_price,
       item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
ORDER BY o.order_date DESC LIMIT 15
''', label="4b. unnest + join")


Full rollup: orders → line_items → products → customers, one query.

In [ ]:
run_query(f'''
SELECT c.region, p.category, SUM(item.quantity * item.unit_price) AS total_revenue
FROM {TBL("orders")} o, UNNEST(o.line_items) AS item
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id
JOIN {TBL("products")} p ON item.product_id = p.product_id
WHERE o.status = 'Completed'
GROUP BY c.region, p.category
ORDER BY total_revenue DESC
''', label="4c. revenue rollup")



> 🖥️ Run 4c in the console, then check Execution details — each join and the UNNEST show up as
> separate stages with rows-in/rows-out, useful for spotting which join is actually expensive.



## 5. Views
A view stores the query, not data — every read re-runs it live. Wraps 4c's join so analysts don't
retype it.


In [ ]:
run_query(f'''
CREATE OR REPLACE VIEW {TBL("v_completed_order_details")} AS
SELECT o.order_id, o.order_date, o.region, c.customer_segment, item.product_name,
       item.quantity, item.unit_price, item.quantity * item.unit_price AS line_total
FROM {TBL("orders")} o
JOIN {TBL("customers")} c ON o.customer_id = c.customer_id,
UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
''', label="create view", expect_rows=False)

run_query(f'SELECT region, SUM(line_total) AS revenue FROM {TBL("v_completed_order_details")} GROUP BY region',
          label="query the view")



> 🖥️ `v_completed_order_details` shows a "view" icon; Details tab shows the stored SQL, no rows
> of its own. Free, always live, zero performance win — every read redoes the full join. That's
> what a materialized view fixes.


## 6. Materialized views

In [ ]:
run_query(f'''
CREATE OR REPLACE MATERIALIZED VIEW {TBL("mv_region_daily_revenue")} AS
SELECT o.region, o.order_date, SUM(item.quantity * item.unit_price) AS daily_revenue,
       APPROX_COUNT_DISTINCT(o.order_id) AS num_orders
FROM {TBL("orders")} o, UNNEST(o.line_items) AS item
WHERE o.status = 'Completed'
GROUP BY o.region, o.order_date
''', label="create MV", expect_rows=False)

run_query(f'''
SELECT * FROM {TBL("mv_region_daily_revenue")} WHERE region = 'APAC' ORDER BY order_date DESC
''', label="query the MV")



Two BigQuery restrictions that bite people here, worth knowing before you hit them: incremental
MVs can't use `COUNT(DISTINCT ...)` — hence `APPROX_COUNT_DISTINCT` above — and a materialized
view must live in the *same project* as every table it references (a plain view has no such
restriction). Not an issue here since `orders` is already in this project.

> 🖥️ Distinct "materialized view" icon; Details tab has a Refresh section a plain view never
> shows, since it has nothing to refresh.



## 7. Partitioning — proving it actually matters
`orders` is already `PARTITION BY order_date, CLUSTER BY region, status`. Here's why: an
unpartitioned copy for comparison.


In [ ]:
run_query(f'CREATE OR REPLACE TABLE {TBL("orders_unpartitioned")} AS SELECT * FROM {TBL("orders")}',
          label="unpartitioned copy", expect_rows=False)


In [ ]:
print("partitioned:")
run_query(f"SELECT * FROM {TBL('orders')} WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'",
          label="filtered")
print("\nunpartitioned:")
run_query(f"SELECT * FROM {TBL('orders_unpartitioned')} WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31'",
          label="filtered")



At this table's tiny size the gap looks unremarkable — the same pruning is what turns a 500GB
scan into a 2GB one at real scale. Stacking clustering on top narrows further within a partition:


In [ ]:
run_query(f'''
SELECT * FROM {TBL("orders")}
WHERE order_date BETWEEN '2026-01-01' AND '2026-01-31' AND region = 'APAC' AND status = 'Completed'
''', label="partition + cluster pruning")



> 🖥️ `orders`'s Details tab shows both a Partition section (`DATE(order_date)`) and a Clustering
> section (`region, status`) — the one place both show up on the same table.



## 8. External tables, BigLake & Lakehouse for Apache Iceberg
Some of ShopIndex's data is owned by other teams' pipelines — copying it in means paying for
duplicate storage that goes stale the moment the source updates. External tables query it in
place instead.


In [ ]:
run_query(f'''
CREATE OR REPLACE EXTERNAL TABLE {TBL("orders_external")}
OPTIONS (format = 'NEWLINE_DELIMITED_JSON', uris = ['{orders_uri}'])
''', label="create external table", expect_rows=False)

run_query(f'SELECT order_id, order_date, status FROM {TBL("orders_external")} LIMIT 10',
          label="query it")



> 🖥️ `orders_external`'s Details tab shows Table type: EXTERNAL and the source URI — no data
> copied. Compare Storage stats (~0 bytes) against `orders`'s.

Without `WITH CONNECTION`, this is a plain external table — query-time IAM only, no fine-grained
access control. Adding a connection turns it into a BigLake table with the same row/column
security as native tables.

### Managed Iceberg — a step further
A managed Iceberg table is a real BigQuery-managed table that happens to store its data as
Parquet + Iceberg metadata in *your own* bucket — full BigQuery DML/time-travel/access-control,
while staying portable to Spark/Trino. Needs a one-time Cloud resource connection first.


In [ ]:
import json, time

if RUN_ICEBERG_STEP:
    ICEBERG_CONNECTION = f"{PROJECT_ID}.{LOCATION}.iceberg_conn"

    existing = !bq show --connection {ICEBERG_CONNECTION} 2>&1
    if any(x in "\n".join(existing).lower() for x in ["not found", "notfound"]):
        !bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} \
            --location={LOCATION} iceberg_conn
        print("created connection")
    else:
        print("connection already exists, reusing it")

    raw = !bq show --format=prettyjson --connection {ICEBERG_CONNECTION}
    connection_sa = json.loads("\n".join(raw))["cloudResource"]["serviceAccountId"]
    print("connection service account:", connection_sa)

    !gsutil iam ch serviceAccount:{connection_sa}:roles/storage.objectAdmin gs://{BUCKET_NAME}
    print("granted storage.objectAdmin, waiting 45s for IAM to propagate...")
    time.sleep(45)

    run_query(f'''
    CREATE OR REPLACE TABLE {TBL("orders_iceberg")} (
      order_id INT64, customer_id INT64, region STRING, order_date DATE, status STRING
    )
    WITH CONNECTION `{ICEBERG_CONNECTION}`
    OPTIONS (
      file_format = 'PARQUET', table_format = 'ICEBERG',
      storage_uri = 'gs://{BUCKET_NAME}/{DATASET_ID}/iceberg_warehouse/orders_iceberg'
    )
    ''', label="create managed Iceberg table", expect_rows=False)

    run_query(f'''
    INSERT INTO {TBL("orders_iceberg")}
    SELECT order_id, customer_id, region, order_date, status FROM {TBL("orders")}
    ''', label="populate it", expect_rows=False)

    run_query(f'SELECT region, COUNT(*) AS num_orders FROM {TBL("orders_iceberg")} GROUP BY region ORDER BY num_orders DESC',
              label="query it")
else:
    print("RUN_ICEBERG_STEP is False — flip it above to run this live.")



> 🖥️ `orders_iceberg`'s Details tab shows a BigLake badge and Storage format: ICEBERG. Your
> bucket's `iceberg_warehouse/` prefix now has real Parquet + metadata files, readable directly by
> Spark or Trino, outside BigQuery entirely.



## 9. Pricing — on-demand vs. reservations & slot commitments
Every query in this notebook has been on-demand billing (pay per byte scanned) — the default for
every new project, and the right choice for anything spiky/moderate-volume like this lab. The
alternative, reservations, buys a committed pool of slots for predictable cost at steady,
high-volume production scale.


In [ ]:
from google.cloud import bigquery_reservation_v1

reservation_client = bigquery_reservation_v1.ReservationServiceClient()
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}"

commitments = list(reservation_client.list_capacity_commitments(parent=parent))
reservations = list(reservation_client.list_reservations(parent=parent))

print(f"capacity commitments: {len(commitments)}")
print(f"reservations: {len(reservations)}")
if not commitments and not reservations:
    print("\nboth empty -- you're on pure on-demand pricing, same as this whole notebook so far.")



Actually purchasing a reservation is real, ongoing money (annual/3-year commitments, Enterprise+
edition required) — not something to run live in a training session. The commands, for reference:


In [ ]:
if RUN_RESERVATION_STEP:
    print(f'''
# purchase a commitment (real money, annual/3-year lock-in):
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --capacity_commitment=true \\
#   --edition=ENTERPRISE --plan=ANNUAL --slots=100

# create a reservation drawing from it:
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --reservation=true \\
#   --slots=100 --edition=ENTERPRISE my-reservation

# assign this project to it (without this, it stays on-demand):
# bq mk --project_id={PROJECT_ID} --location={LOCATION} --reservation_assignment=true \\
#   --reservation_id=my-reservation --job_type=QUERY --assignee_type=PROJECT --assignee_id={PROJECT_ID}
''')
else:
    print("RUN_RESERVATION_STEP is False -- reference only by design, see above.")



> 🖥️ BigQuery > Administration > Workload management → Slot reservations and Commitments tabs
> show exactly what the calls above returned, and are where you'd actually click "Buy slots."



## 10. IAM & data access control
Before row-level or column-level security matter at all, there's a coarser question: who can see
this dataset or table in the first place? IAM works at project, dataset, and table/view level.

Common gotcha worth knowing: `roles/bigquery.dataViewer` lets someone read data, but
`roles/bigquery.jobUser` is what actually lets them run a query job — granting one without the
other is a frequent "why can't this user query anything" moment.


In [ ]:
dataset = bq_client.get_dataset(f"{PROJECT_ID}.{DATASET_ID}")
print(f"access entries on {DATASET_ID}:")
for entry in dataset.access_entries:
    print(f"  {entry.entity_type}: {entry.entity_id} -> {entry.role or '(implied)'}")


Grant read-only access to just one view — not the whole dataset:

In [ ]:
STUDENT_EMAIL = "student1@yourdomain.com"  # <-- set to a real user/group before uncommenting the grant

table_ref = bq_client.get_table(TBL("v_completed_order_details").strip("`"))
policy = bq_client.get_iam_policy(table_ref)
policy.bindings.append({"role": "roles/bigquery.dataViewer", "members": {f"user:{STUDENT_EMAIL}"}})
# bq_client.set_iam_policy(table_ref, policy)  # uncomment once STUDENT_EMAIL is real

print(f"would grant dataViewer on v_completed_order_details to {STUDENT_EMAIL} (inert until uncommented)")



> 🖥️ The view's Sharing > Permissions panel shows the same bindings in the console's role-picker
> UI — how this is actually managed day to day.

IAM, row-level security (next), and column masking (Section 15) stack: IAM gates whether you can
query at all, RLS filters which rows you see, masking decides whether specific columns show real
or obscured values.



## 11. BigQuery ML — four questions, one SQL pattern
Train, evaluate, predict — same three steps regardless of model type, just the SQL and the
metric columns that change.


**11.1 Will this customer churn?** — logistic regression, binary classification.

In [ ]:
if RUN_BQML_STEP:
    run_query(f'''
    CREATE OR REPLACE MODEL {TBL("churn_model")}
    OPTIONS(model_type='logistic_reg', input_label_cols=['churned']) AS
    SELECT c.customer_segment, c.region, COUNT(o.order_id) AS num_orders,
           IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value, c.churned
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.churned, c.customer_id
    ''', label="train churn_model", expect_rows=False)

    run_query(f'SELECT * FROM ML.EVALUATE(MODEL {TBL("churn_model")})', label="evaluate")
else:
    print("RUN_BQML_STEP is False.")


**11.2 What's this customer worth over time?** — linear regression, continuous label.

In [ ]:
if RUN_BQML_STEP:
    run_query(f'''
    CREATE OR REPLACE MODEL {TBL("ltv_model")}
    OPTIONS(model_type='linear_reg', input_label_cols=['lifetime_value']) AS
    SELECT c.customer_segment, c.region,
           DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days,
           IFNULL(SUM(item.quantity * item.unit_price), 0) AS lifetime_value
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_segment, c.region, c.signup_date, c.customer_id
    ''', label="train ltv_model", expect_rows=False)

    run_query(f'SELECT * FROM ML.EVALUATE(MODEL {TBL("ltv_model")})',
              label="evaluate — r2_score and mean_absolute_error, not precision/recall")

    run_query(f'''
    SELECT customer_segment, region, tenure_days, predicted_lifetime_value
    FROM ML.PREDICT(MODEL {TBL("ltv_model")}, (
      SELECT c.customer_segment, c.region, DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
      FROM {TBL("customers")} c LIMIT 10
    ))
    ''', label="predict for 10 customers")
else:
    print("RUN_BQML_STEP is False.")


**11.3 Do customers fall into natural groups?** — KMEANS, unsupervised, no label.

In [ ]:
if RUN_BQML_STEP:
    run_query(f'''
    CREATE OR REPLACE MODEL {TBL("customer_segments_model")}
    OPTIONS(model_type='kmeans', num_clusters=3) AS
    SELECT COUNT(o.order_id) AS num_orders,
           IFNULL(SUM(item.quantity * item.unit_price), 0) AS total_spend,
           DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
    FROM {TBL("customers")} c
    LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
    LEFT JOIN UNNEST(o.line_items) AS item
    GROUP BY c.customer_id, c.signup_date
    ''', label="train customer_segments_model", expect_rows=False)

    run_query(f'SELECT centroid_id, feature, numerical_value FROM ML.CENTROIDS(MODEL {TBL("customer_segments_model")}) ORDER BY centroid_id, feature',
              label="the 3 centroids")

    # note: KMEANS output is CENTROID_ID, not predicted_<label> -- there's no label to predict from,
    # so the "predicted_..." naming convention supervised models use doesn't apply here
    run_query(f'''
    SELECT c.customer_id, c.customer_segment, CENTROID_ID
    FROM ML.PREDICT(MODEL {TBL("customer_segments_model")}, (
      SELECT c.customer_id, COUNT(o.order_id) AS num_orders,
             IFNULL(SUM(item.quantity * item.unit_price), 0) AS total_spend,
             DATE_DIFF(CURRENT_DATE(), c.signup_date, DAY) AS tenure_days
      FROM {TBL("customers")} c
      LEFT JOIN {TBL("orders")} o ON c.customer_id = o.customer_id
      LEFT JOIN UNNEST(o.line_items) AS item
      GROUP BY c.customer_id, c.signup_date
    )) AS predicted
    JOIN {TBL("customers")} c ON predicted.customer_id = c.customer_id
    ORDER BY CENTROID_ID
    ''', label="which cluster each customer falls into")
else:
    print("RUN_BQML_STEP is False.")


**11.4 What should revenue look like next?** — ARIMA_PLUS, time series forecasting.

In [ ]:
if RUN_BQML_FORECAST_STEP:
    run_query(f'''
    CREATE OR REPLACE MODEL {TBL("revenue_forecast_model")}
    OPTIONS(model_type='ARIMA_PLUS', time_series_timestamp_col='order_date',
            time_series_data_col='daily_revenue', auto_arima=TRUE) AS
    SELECT o.order_date, SUM(item.quantity * item.unit_price) AS daily_revenue
    FROM {TBL("orders")} o, UNNEST(o.line_items) AS item
    WHERE o.status = 'Completed'
    GROUP BY o.order_date
    ''', label="train revenue_forecast_model", expect_rows=False)

    run_query(f'SELECT * FROM ML.ARIMA_EVALUATE(MODEL {TBL("revenue_forecast_model")})', label="candidate ARIMA models")

    run_query(f'''
    SELECT forecast_timestamp, forecast_value, prediction_interval_lower_bound, prediction_interval_upper_bound
    FROM ML.FORECAST(MODEL {TBL("revenue_forecast_model")}, STRUCT(14 AS horizon, 0.95 AS confidence_level))
    ORDER BY forecast_timestamp
    ''', label="14-day forecast")
else:
    print("RUN_BQML_FORECAST_STEP is False.")
    print("(our order history is only a few months -- nowhere near the multiple full seasonal")
    print("cycles ARIMA_PLUS really wants, so treat this as 'seeing the mechanism work', not a")
    print("forecast to actually trust)")



> 🖥️ Each model's Evaluation tab renders the relevant chart automatically — ROC curve for the
> classifiers, a cluster plot for KMEANS, a forecast with confidence bands for ARIMA — no
> charting code needed on your end.



## 12. Time travel & change history
Someone ran a bad UPDATE, or dropped a table, 20 minutes ago — can the old state come back
without a separate backup system?


In [ ]:
run_query(f"SELECT order_id, status FROM {TBL('orders')} WHERE order_id = 1001", label="before")
run_query(f"UPDATE {TBL('orders')} SET status = 'Cancelled' WHERE order_id = 1001", label="update", expect_rows=False)
run_query(f"SELECT order_id, status FROM {TBL('orders')} WHERE order_id = 1001", label="after")


In [ ]:
# query the table as it looked before the update above -- no restore needed
run_query(f'''
SELECT order_id, status FROM {TBL("orders")}
FOR SYSTEM_TIME AS OF TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 5 MINUTE)
WHERE order_id = 1001
''', label="time travel")



Whole-table recovery (not just a row) works differently — there's no `UNDROP TABLE` statement in
GoogleSQL. The actual mechanism is a **copy job** sourcing from the table's pre-deletion snapshot
decorator (`table@timestamp_ms`); the query engine itself can't read that decorator off an
already-deleted table, only `copy_table()` can.


In [ ]:
import time
from google.cloud.bigquery import TableReference

run_query(f'CREATE OR REPLACE TABLE {TBL("scratch_table")} AS SELECT * FROM {TBL("products")} LIMIT 5',
          label="create scratch table", expect_rows=False)
time.sleep(10)  # let the write fully commit before treating any timestamp as a valid snapshot point
snapshot_ms = int(time.time() * 1000)

run_query(f'DROP TABLE {TBL("scratch_table")}', label="drop it (the accident)", expect_rows=False)

run_query(f'''
SELECT table_name FROM `{PROJECT_ID}.{DATASET_ID}.INFORMATION_SCHEMA.TABLES`
WHERE table_name = 'scratch_table'
''', label="confirm it's gone")

source_ref = TableReference.from_string(f"{PROJECT_ID}.{DATASET_ID}.scratch_table@{snapshot_ms}")
dest_ref = TableReference.from_string(f"{PROJECT_ID}.{DATASET_ID}.scratch_table")
bq_client.copy_table(source_ref, dest_ref).result()
print("recovered via copy job from the snapshot decorator")

run_query(f'SELECT * FROM {TBL("scratch_table")}', label="confirm the data is back")



## 13. Vector Search for RAG
A support chatbot needs to find semantically similar products from a natural-language query —
meaning, not keyword matching. Needs a remote connection to a Vertex AI embedding model first.


In [ ]:
import json, time
from google.api_core.exceptions import BadRequest, NotFound

if RUN_VECTOR_SEARCH_STEP:
    !gcloud services enable aiplatform.googleapis.com bigqueryconnection.googleapis.com --project={PROJECT_ID} --quiet

    CONNECTION_NAME = "embedding_conn"
    existing = !bq show --connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME} 2>&1
    if any(x in "\n".join(existing).lower() for x in ["not found", "notfound"]):
        !bq mk --connection --connection_type=CLOUD_RESOURCE --project_id={PROJECT_ID} --location={LOCATION} {CONNECTION_NAME}
    else:
        print("connection already exists, reusing it")

    raw = !bq show --format=prettyjson --connection {PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}
    embedding_sa = json.loads("\n".join(raw))["cloudResource"]["serviceAccountId"]
    print("connection service account:", embedding_sa)

    !gcloud projects add-iam-policy-binding {PROJECT_ID} --member="serviceAccount:{embedding_sa}" --role="roles/aiplatform.user" --quiet --condition=None

    CONNECTION_ID = f"{PROJECT_ID}.{LOCATION}.{CONNECTION_NAME}"

    # IAM/API propagation timing isn't predictable -- retry the actual operation instead of
    # guessing a fixed sleep duration
    for attempt in range(1, 7):
        try:
            run_query(f'''
            CREATE OR REPLACE MODEL {TBL("embedding_model")}
            REMOTE WITH CONNECTION `{CONNECTION_ID}`
            OPTIONS (endpoint = 'text-embedding-004')
            ''', label=f"create embedding model (attempt {attempt})", expect_rows=False)
            print("embedding model ready")
            break
        except (BadRequest, NotFound) as e:
            wait = min(15 * attempt, 90)
            print(f"attempt {attempt} failed, retrying in {wait}s: {str(e)[:120]}")
            if attempt == 6:
                raise
            time.sleep(wait)
else:
    print("RUN_VECTOR_SEARCH_STEP is False.")


In [ ]:
if RUN_VECTOR_SEARCH_STEP:
    run_query(f'''
    CREATE OR REPLACE TABLE {TBL("product_embeddings")} AS
    SELECT product_id, product_name, ml_generate_embedding_result AS embedding
    FROM ML.GENERATE_EMBEDDING(
      MODEL {TBL("embedding_model")},
      (SELECT product_id, product_name, description AS content FROM {TBL("products")})
    )
    ''', label="generate embeddings", expect_rows=False)

    # CREATE VECTOR INDEX with IVF requires at least 5,000 rows -- it's a hard minimum, not a
    # graceful fallback, and this table (26 products) is nowhere close. Below that threshold,
    # BigQuery's own error message says to skip the index and call VECTOR_SEARCH directly --
    # it just does a brute-force scan instead, which is completely fine at this size.
    run_query(f'''
    SELECT base.product_name, distance
    FROM VECTOR_SEARCH(
      TABLE {TBL("product_embeddings")}, 'embedding',
      (SELECT embedding FROM {TBL("product_embeddings")} WHERE product_name = 'Wireless Mouse'),
      top_k => 5
    )
    ''', label="find similar products (brute-force, no index needed at this size)")
else:
    print("RUN_VECTOR_SEARCH_STEP is False.")



> 🖥️ `product_embeddings`'s Schema tab shows `embedding` as an ordinary `FLOAT64` array — worth
> pointing out it's not a special vector type. There's no index to see in the console at this
> table's size (26 products) -- `CREATE VECTOR INDEX` with IVF has a hard 5,000-row minimum, so
> we skip it entirely and let `VECTOR_SEARCH` run as a brute-force scan instead, which is exactly
> what BigQuery recommends below that threshold.



## 14. Remote functions & connections
BigQuery's built-in functions can't call an arbitrary external service — a currency API, an
internal validation rule. Remote functions bridge SQL to an HTTP endpoint (Cloud Functions/Cloud
Run), reusing the same connection mechanism as Sections 8 and 13.

Needs a real deployed endpoint, so this stays reference-only:


In [ ]:
print(f'''
-- 1. bq mk --connection --connection_type=CLOUD_RESOURCE --location={LOCATION} remote_conn
-- 2. grant the connection's service account Cloud Functions Invoker on your endpoint

CREATE OR REPLACE FUNCTION {TBL("convert_to_usd")}(amount FLOAT64, currency STRING)
RETURNS FLOAT64
REMOTE WITH CONNECTION `{PROJECT_ID}.{LOCATION}.remote_conn`
OPTIONS (endpoint = 'https://YOUR_CLOUD_FUNCTION_URL')

-- then call it like any other function:
-- SELECT order_id, convert_to_usd(total_amount, 'INR') AS amount_usd FROM orders
''')



> 🖥️ SQL workspace > External connections lists every connection this notebook created (Iceberg,
> vector search, remote functions) in one place, with each one's service account email.



## 15. Data lineage
"Where did this number in `mv_region_daily_revenue` actually come from?" Every CTAS/view/MV we've
built has an implicit dependency chain — capture is automatic once the Data Lineage API is
enabled (done in Section 0), no code required to generate it, only to query it back.


In [ ]:
from google.cloud import datacatalog_lineage_v1

lineage_client = datacatalog_lineage_v1.LineageClient()
target = f"//bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/v_completed_order_details"

request = datacatalog_lineage_v1.SearchLinksRequest(
    parent=f"projects/{PROJECT_ID}/locations/{LOCATION.lower()}",
    target=datacatalog_lineage_v1.EntityReference(fully_qualified_name=target),
)

try:
    links = list(lineage_client.search_links(request=request))
    if links:
        for link in links:
            print(f"{link.source.fully_qualified_name}  ->  {link.target.fully_qualified_name}")
    else:
        print("nothing yet -- lineage can take a few minutes up to 24 hours to appear, not instant")
except Exception as e:
    print(f"lineage query failed (usually just means nothing's propagated yet): {e}")



> 🖥️ The view's Lineage tab shows a visual dependency graph — far more legible than the raw API
> output above, and the tab to actually reach for when demoing this live.



## 16. Row-level security
The APAC team should only see APAC orders on the *same* `orders` table everyone else queries —
duplicating the table per region doesn't scale and drifts out of sync.


In [ ]:
import requests, google.auth, google.auth.transport.requests

# reliable whoami -- gcloud config get-value account is flaky under Colab's auth flow
credentials, _ = google.auth.default()
credentials.refresh(google.auth.transport.requests.Request())
me = requests.get("https://www.googleapis.com/oauth2/v1/userinfo",
                   headers={"Authorization": f"Bearer {credentials.token}"}).json()["email"]

if RUN_RLS_STEP:
    run_query(f'''
    CREATE OR REPLACE ROW ACCESS POLICY apac_only
    ON {TBL("orders")}
    GRANT TO ('user:{me}')
    FILTER USING (region = 'APAC')
    ''', label="create RLS policy", expect_rows=False)
    print(f"policy created against your own account ({me}) -- query orders now, you'll only see APAC rows")
else:
    print("RUN_RLS_STEP is False.")



> 🖥️ `orders`'s Details tab → Row access policies section lists `apac_only` with its filter and
> granted principals, visible without needing to query as the restricted user.



## 17. Encryption — AEAD & dynamic data masking
Two different questions: can we encrypt a column so even someone with table access can't read it
without a key (AEAD)? And can most users see a masked version while a few see the real value
(dynamic masking)?


In [ ]:
run_query(f'''
DECLARE keyset BYTES DEFAULT KEYS.NEW_KEYSET('AEAD_AES_GCM_256');
SELECT customer_id, AEAD.ENCRYPT(keyset, email, CAST(customer_id AS STRING)) AS encrypted_email
FROM {TBL("customers")} LIMIT 10
''', label="AEAD encryption")



The keyset above is generated inline and discarded when the query ends — real usage stores it in
Cloud KMS so encrypted values can actually be decrypted later.

Dynamic masking is the alternative when most users should just never see raw sensitive values by
default — built on policy tags (same taxonomy mechanism as the governance notebook), reference
only here since it needs a taxonomy already set up:


In [ ]:
print(f'''
ALTER TABLE {TBL("customers")}
ALTER COLUMN email
SET OPTIONS (policy_tags = ['projects/{PROJECT_ID}/locations/{LOCATION}/taxonomies/TAXONOMY_ID/policyTags/POLICY_TAG_ID']);
-- masking rule (EMAIL_MASK, DEFAULT_MASKING_VALUE, etc.) then set on the policy tag itself,
-- via console or the Data Policy API -- unauthorized queries then just see XXXXX@XXXXX.com
''')



> 🖥️ Governance > Knowledge Catalog > Policy tags → a tag's Data masking tab picks a rule
> directly — easier to configure visually than via API for a one-off classification.

AEAD for genuine reversible encryption (compliance, right-to-be-forgotten via key deletion);
masking when most users should never see raw values by default with a small group unmasked.


## Cleanup

In [ ]:
CONFIRM_DELETE = False  #@param {type:"boolean"}

if not CONFIRM_DELETE:
    print("CONFIRM_DELETE is False -- nothing happens. Flip it and re-run when ready.")


In [ ]:
if CONFIRM_DELETE:
    try:
        bq_client.delete_dataset(f"{PROJECT_ID}.{DATASET_ID}", delete_contents=True, not_found_ok=True)
        print(f"deleted {DATASET_ID} (every table, view, MV, and model inside it)")
    except Exception as e:
        print("dataset cleanup skipped/failed:", e)

    try:
        bucket.delete(force=True)
        print(f"deleted bucket {BUCKET_NAME}")
    except Exception as e:
        print("bucket cleanup skipped/failed:", e)

    for conn in ["iceberg_conn", "embedding_conn", "remote_conn"]:
        try:
            !bq rm --connection --project_id={PROJECT_ID} --location={LOCATION} -f {conn}
            print("deleted connection:", conn)
        except Exception as e:
            print(f"{conn} cleanup skipped (may not exist): {e}")

    if RUN_RESERVATION_STEP:
        print("\nif you actually purchased a reservation/commitment (the Section 9 commands were")
        print("commented out by default), tear down in order: assignment -> reservation -> commitment.")
        print("commitments can't be deleted before their term ends.")

    print("\ncleanup complete")



> 🖥️ SQL workspace, Cloud Storage, and External connections should all show nothing left from
> this notebook — and Workload management should show no reservations unless you deliberately
> kept one.
